In [1]:
pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 7.1 MB/s eta 0:00:00


In [2]:
from transformers import BertTokenizer, BartForConditionalGeneration, Text2TextGenerationPipeline

tokenizer = BertTokenizer.from_pretrained("fnlp/bart-base-chinese")
model = BartForConditionalGeneration.from_pretrained("fnlp/bart-base-chinese")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/259k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/561M [00:00<?, ?B/s]

In [3]:
import pandas as pd
from transformers import BertTokenizer
from datasets import Dataset

train_df = pd.read_csv("train.csv")
train_df

,ques_content,ques_knowledges
0,题目内容以下关于误差的描述哪项是不正确的选项A实验中的错误被称为误差B误差是由于不严格遵守操...,误差与减小误差的方法
1,题目内容水稻收割后将带壳的稻谷堆成一个圆锥形谷堆放在场地上物理老师和数学老师给学生布置了一项...,质量的测量
2,题目内容某油库有密度为083103kgm3的石油275m3需运往外地目前有若干辆载油量为25...,密度公式的应用
3,某日在一条城市道路上两辆轿车发生追尾碰撞事故交警调查时前车司机表示我的车速很慢后面的车突然加...,惯性的利用及危害防止
4,问题为什么在砍刀的刀柄上会有凹凸不平的花纹设计,增大摩擦的方法及生活中的实例
...,...,...
698,题目内容以下哪个单位换算是正确的选项A20cm20001m020mB20cm20100cm0...,长度的单位及换算
699,题目内容在使用刻度尺测量长度时下列哪个做法是错误的选项A测量时应确保刻度尺没有倾斜B测量过程...,刻度尺的使用
700,题目内容当工人使用铲子将煤送入炉膛时铲子本身并不会进入炉膛而是煤能够飞入这是什么原因,惯性的利用及危害防止
701,题目内容一块石碑的长度宽度和高度分别是text20mtext12m和mathrm8m为计算其...,密度公式的应用


In [4]:
import pandas as pd
from transformers import BertTokenizer
from datasets import Dataset

test_df = pd.read_csv("test.csv")

# 加载分词器
tokenizer = BertTokenizer.from_pretrained("fnlp/bart-base-chinese")

# 数据预处理函数
def preprocess_function(examples):
    inputs = examples["ques_content"]
    targets = examples["ques_knowledges"]

    # 对输入和目标进行编码
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 将数据转换为 Hugging Face 数据集
train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True)
test_dataset = Dataset.from_pandas(test_df).map(preprocess_function, batched=True)

Map:   0%|          | 0/703 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3961: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/176 [00:00<?, ? examples/s]

In [5]:
from transformers import BartForConditionalGeneration, Trainer, TrainingArguments

# 加载模型
model = BartForConditionalGeneration.from_pretrained("fnlp/bart-base-chinese")

# 训练参数
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# 开始训练
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 1830695178 (1830695178-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.041300,0.035398
2,0.025000,0.026974
3,0.011800,0.025368
4,0.010700,0.025233
5,0.010800,0.024375


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_eos_token_id': 102}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=440, training_loss=0.26219412064687775, metrics={'train_runtime': 410.7573, 'train_samples_per_second': 8.557, 'train_steps_per_second': 1.071, 'total_flos': 1071611825356800.0, 'train_loss': 0.26219412064687775, 'epoch': 5.0})

In [6]:
import os
from transformers import BartForConditionalGeneration

# 获取最新的 checkpoint 目录
results_dir = "./results"
latest_checkpoint = max(
    [os.path.join(results_dir, d) for d in os.listdir(results_dir) if d.startswith("checkpoint")],
    key=os.path.getmtime
)

# 加载最新的 checkpoint
model = BartForConditionalGeneration.from_pretrained(latest_checkpoint)
print(f"成功加载模型：{latest_checkpoint}")

成功加载模型：./results/checkpoint-440


In [7]:
from transformers import BertTokenizer, BartForConditionalGeneration

# 加载分词器
tokenizer = BertTokenizer.from_pretrained("fnlp/bart-base-chinese")

In [8]:
model.eval()

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(51271, 768, padding_idx=0)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(51271, 768, padding_idx=0)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_lay

In [9]:
def predict(text):
    # 对输入进行编码（移除 token_type_ids）
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)

    # 只保留 model 需要的参数
    inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}

    # 生成预测
    output_ids = model.generate(**inputs, max_length=128, num_beams=5)

    # 解码预测结果并删除空格
    predicted_text = tokenizer.decode(output_ids[0], skip_special_tokens=True).replace(" ", "")
    return predicted_text

In [12]:
test_text = "问题描述以下哪一项单位换算是正确的选项A45厘米45厘米001米045米B45厘米4501045米C45厘米45001米045米D45厘米45001045米"
prediction = predict(test_text)
print("预测的知识点类型:", prediction)

预测的知识点类型: 长度的单位及换算


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score

test_df = pd.read_csv("test.csv")

# 进行批量预测
test_df["predicted_ques_knowledges"] = test_df["ques_content"].apply(predict)

# 计算准确率
y_true = test_df["ques_knowledges"].astype(str)  # 真实标签
y_pred = test_df["predicted_ques_knowledges"].astype(str)  # 预测标签

accuracy = accuracy_score(y_true, y_pred)  # 计算准确率

print(f"模型在测试集上的准确率: {accuracy:.4f}")

# 保存预测结果
test_df.to_csv("predicted_test.csv", index=False)